In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.path as mpltPath
import pandas as pd
import random
import cloudpickle as cp
from chainconsumer import ChainConsumer

import torch
import torch.nn as nn

from sbi import __version__ as sbi_version
from sbi.inference import SNLE
from sbi.utils import BoxUniform
from sbi import utils
from sbi.utils.get_nn_models import posterior_nn

from src.load_data import load_data_wide, load_data_narrow
from hyperparameters import hparams

from torch import from_numpy

# Set random seeds for reproducibility
RANDOM_SEED = 42

# Set Python random seed
random.seed(RANDOM_SEED)

# Set NumPy random seed
np.random.seed(RANDOM_SEED)

# Set PyTorch random seed
torch.manual_seed(RANDOM_SEED)

# Set CUDA random seed if available
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)
    # Make CUDA operations deterministic (may impact performance)
    #torch.backends.cudnn.deterministic = True
    #torch.backends.cudnn.benchmark = False

# Set environment variable for additional reproducibility
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print(f"SBI version: {sbi_version}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA deterministic: {torch.backends.cudnn.deterministic}")
print(f"Random seed set to: {RANDOM_SEED}")


SBI version: 0.21.0
Using device: cuda
GPU name: NVIDIA A40
GPU memory: 47.7 GB
CUDA deterministic: False
Random seed set to: 42


In [3]:
training_parameters, training_data = load_data_wide(hparams)
training_parameters_narrow, training_data_narrow = load_data_narrow(hparams)

OutOfMemoryError: CUDA out of memory. Tried to allocate 6.71 GiB (GPU 0; 44.43 GiB total capacity; 6.99 GiB already allocated; 3.51 GiB free; 6.99 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
training_parameters_mean = training_parameters.mean(0)
training_parameters_std = training_parameters.std(0)
training_parameters_stand = (training_parameters - training_parameters_mean) / training_parameters_std
training_parameters_stand_narrow = (training_parameters_narrow - training_parameters_mean) / training_parameters_std

In [1]:
from torch.utils.data import TensorDataset, DataLoader
#train_dataset = TensorDataset(training_parameters_stand, training_data)
#train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
#train_dataset_narrow = TensorDataset(training_parameters_stand_narrow, training_data_narrow)
#train_loader_narrow = DataLoader(train_dataset_narrow, batch_size=64, shuffle=False)

import cloudpickle as cpk
with open(f'/project/ag-weller/Kai.Lehman/C3NN_Models/MSE_Models/3convs_bandwidth2_p63.cpk', 'rb') as f:
    model = cpk.load(f)

#model.to('cuda')

#training_data.to('cuda')
#training_data_narrow.to('cuda')
print('forward...')
#y_pred_stand = model(training_data)
#training_data.to('cpu')
#torch.cuda.empty_cache()
#y_pred_stand_narrow = model(training_data_narrow)

forward...


In [5]:
#y_pred = y_pred_stand *training_parameters_std + training_parameters_mean
#y_pred_narrow = y_pred_stand_narrow *training_parameters_std + training_parameters_mean

In [6]:
#with open(f"/project/ag-weller/Kai.Lehman/C3NN_Models/MSE_Models/3convs_bandwidth2_p63_wide_summary.cpk", "wb") as handle:
#    cpk.dump(y_pred, handle)
#with open(f"/project/ag-weller/Kai.Lehman/C3NN_Models/MSE_Models/3convs_bandwidth2_p63_narrow_summary.cpk", "wb") as handle:
#    cpk.dump(y_pred_narrow, handle)
#sys.exit()

In [ ]:
with open(f'/project/ag-weller/Kai.Lehman/C3NN_Models/MSE_Models/3convs_bandwidth2_p63_wide_summary.cpk', 'rb') as f:
    y_pred = cpk.load(f)
with open(f'/project/ag-weller/Kai.Lehman/C3NN_Models/MSE_Models/3convs_bandwidth2_p63_narrow_summary.cpk', 'rb') as f:
    y_pred_narrow = cpk.load(f)

In [8]:
# Load measured 2PCFs
#x_train = np.load('./data/2PCF_wide_realization0_mean.npy').reshape(1250, -1)
#x_train = x_train[:, np.where(np.argmax(x_train, axis=0))[0]] #Remove later

'''
# Load cosmological parameters as training data
param_names = ['Om', 'sigma8', 'w0']
param_names_latex = [r"$\Omega_m$", r"$\sigma_8$", r"$w_0$"]
y_train = np.array([np.load('CosmoGrid_params.npz')[par][:1250] for par in param_names]).T
'''



'''
# Convert to tensors
y_train = torch.tensor(y_pred, dtype=torch.float32).to(device=device)
x_train = torch.tensor(training_parameters, dtype=torch.float32).to(device=device)
'''

'''
# Define prior bounds for the parameters (Om, sigma8, w0) as the min and max of the training data
prior_min = y_train.min(0)[0].flatten()
prior_max = y_train.max(0)[0].flatten()
prior = BoxUniform(low=prior_min, high=prior_max, device=device)

# Define the (irregular) boundaries of the parameter space
boundary_Om_sig8 = np.array([
    [0.1, 1.4],
    [0.2, 1.4],
    [0.5, 0.65],
    [0.5, 0.4],
    [0.3, 0.4],
    [0.1, 0.9],])

boundary_Om_w0 = np.array([
    [.1, -1.1],
    [.5, -2.],
    [.5, -.33],
    [.1, -.33],])

# Functions to check if a point is within the given 2D parameter spaces
validOS = mpltPath.Path(boundary_Om_sig8)
validOW = mpltPath.Path(boundary_Om_w0)

# Function to remove points that are outside the parameter space
def remove_invalid_points(samples):
    in_os = validOS.contains_points(samples[:,(0,1)])
    in_ow = validOW.contains_points(samples[:,(0,2)])
    return samples[in_os & in_ow]
'''


'\n# Define prior bounds for the parameters (Om, sigma8, w0) as the min and max of the training data\nprior_min = y_train.min(0)[0].flatten()\nprior_max = y_train.max(0)[0].flatten()\nprior = BoxUniform(low=prior_min, high=prior_max, device=device)\n\n# Define the (irregular) boundaries of the parameter space\nboundary_Om_sig8 = np.array([\n    [0.1, 1.4],\n    [0.2, 1.4],\n    [0.5, 0.65],\n    [0.5, 0.4],\n    [0.3, 0.4],\n    [0.1, 0.9],])\n\nboundary_Om_w0 = np.array([\n    [.1, -1.1],\n    [.5, -2.],\n    [.5, -.33],\n    [.1, -.33],])\n\n# Functions to check if a point is within the given 2D parameter spaces\nvalidOS = mpltPath.Path(boundary_Om_sig8)\nvalidOW = mpltPath.Path(boundary_Om_w0)\n\n# Function to remove points that are outside the parameter space\ndef remove_invalid_points(samples):\n    in_os = validOS.contains_points(samples[:,(0,1)])\n    in_ow = validOW.contains_points(samples[:,(0,2)])\n    return samples[in_os & in_ow]\n'

In [9]:
y_comb = torch.cat((y_pred.detach(),y_pred_narrow.detach()))
training_parameters_comb = torch.cat((training_parameters,training_parameters_narrow))
print(y_comb.shape,training_parameters_comb.shape)


lower = np.array([0.1, 0.4, -2.0, 0.87, 0.03, 64.0, 12.0, -2.0])
upper = np.array([0.5, 1.4, -0.33, 1.07, 0.06, 82.0, 15.0, 2.0])
prior = utils.BoxUniform(low=from_numpy(lower), high=from_numpy(upper),device=hparams.device)

# Define the MAF density estimator
density_estimator = posterior_nn(
    model='maf',
    num_transforms=4,         # number of MADEs
    num_blocks=3,             # number of layers in each MADE
    hidden_features=128,      # number of hidden units per layer
)

# Set up SNPE_C with the MAF as density estimator and the compressor as embedding network
#inference = SNPE_C(prior=prior, show_progress_bars=True, density_estimator=density_estimator, device=device)
inference = SNLE(prior=prior, show_progress_bars=True, density_estimator=density_estimator, device=device)
# THIS IS SNLE_A, IS THIS CORRECT TO USE?

_ = inference.append_simulations(training_parameters, y_pred.detach()).train(
    training_batch_size=32, 
    learning_rate=1e-4, 
    stop_after_epochs=20,
    max_num_epochs=1024,
    show_train_summary=True
)

#likelihood_estimator = inference.get_density_estimator()

posterior = inference.build_posterior()

torch.Size([2450, 8]) torch.Size([2450, 8])


/home/k/Kai.Lehman/C3NN_inference/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(


 Neural network successfully converged after 101 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 101
        Best validation performance: 18.4804
        -------------------------
        


In [10]:
with open(f"/project/ag-weller/Kai.Lehman/C3NN_Models/MSE_Models/3convs_bandwidth2_p63_NLE_inference.cpk", "wb") as handle:
    cpk.dump(inference, handle)

/home/k/Kai.Lehman/C3NN_inference/sbi/inference/base.py:429: UserWarning: When the inference object is pickled, the behaviour of the loaded object changes in the following two ways: 1) `.train(..., retrain_from_scratch=True)` is not supported. 2) When the loaded object calls the `.train()` method, it generates a new tensorboard summary writer (instead of appending to the current one).
  warn(


In [7]:
testing_data = np.load("/project/ag-weller/Kai.Lehman/C3NN_data/kappa_noiseless_fiducial_cosmology_150_maps_50_by_50pixels_6_by_6deg_no_smooth_4_des_zsbin_gnom_projection.npy")
testing_data = torch.unsqueeze(torch.tensor(testing_data, dtype=torch.float32).to(device='cuda'),0)
testing_summ_stand = model(testing_data)
testing_summ = testing_summ_stand *training_parameters_std + training_parameters_mean
with open(f"/project/ag-weller/Kai.Lehman/C3NN_Models/MSE_Models/3convs_bandwidth2_p63_test_summary.cpk", "wb") as handle:
    cpk.dump(testing_summ, handle)

In [13]:
%matplotlib inline
testing_data = np.load("/project/ag-weller/Kai.Lehman/C3NN_data/kappa_noiseless_fiducial_cosmology_150_maps_50_by_50pixels_6_by_6deg_no_smooth_4_des_zsbin_gnom_projection.npy")
testing_data = torch.unsqueeze(torch.tensor(testing_data, dtype=torch.float32).to(device='cuda'),0)
testing_summ_stand = model(testing_data)
testing_summ = testing_summ_stand *training_parameters_std + training_parameters_mean
print(testing_data.shape)
print(testing_summ.shape)
print(testing_summ[0])
samples = posterior.sample((2,), x=testing_summ[0]).cpu().numpy() #100000
#samples = remove_invalid_points(samples)
print(samples.shape)

torch.Size([1, 150, 4, 50, 50])
torch.Size([1, 8])
tensor([ 2.4789e-01,  8.3911e-01, -9.1752e-01,  9.6310e-01,  4.5750e-02,
         7.2240e+01,  1.3572e+01, -6.7434e-02], device='cuda:0',
       grad_fn=<SelectBackward0>)


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/120 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
sample_c2_nozscorex_bn_25filter = np.load('/project/ag-weller/Kai.Lehman/C3NN_Models/test_chains/25filters_2order_nozscorex_bn.npy')
sample_c3_nozscorex_bn_25filter = np.load('/project/ag-weller/Kai.Lehman/C3NN_Models/test_chains/25filters_3order_nozscorex_bn.npy')
sample_c4_nozscorex_bn_40filter = np.load('/project/ag-weller/Kai.Lehman/C3NN_Models/test_chains/40filters_4order_nozscorex_bn.npy')

In [ ]:
c = ChainConsumer()

#post_df = pd.DataFrame(samples, columns=param_names_latex)
#truth_dict = dict(zip(param_names_latex, y_train[-1]))

c.add_chain(sample_c2_nozscorex_bn_25filter, parameters=["$\Omega_m$", r"$\sigma_8$", r"$w_0$", r"$n_s$", r"$\Omega_b$", r"$H_0$", r"$M_c^0$", r"$\nu$"], name=r'$\{c_1, c_2\}$ '+'25 filters no standardization plus BatchNorm', shade=True, shade_alpha=0.1, color='purple', linestyle='-', linewidth=2.0, kde=False)
c.add_chain(sample_c3_nozscorex_bn_25filter, parameters=["$\Omega_m$", r"$\sigma_8$", r"$w_0$", r"$n_s$", r"$\Omega_b$", r"$H_0$", r"$M_c^0$", r"$\nu$"], name=r'$\{c_1, c_2, c_3\}$ '+'25 filters no standardization plus BatchNorm', shade=True, shade_alpha=0.3, color='b', linestyle='-', linewidth=2.0, kde=False)
c.add_chain(sample_c4_nozscorex_bn_40filter, parameters=["$\Omega_m$", r"$\sigma_8$", r"$w_0$", r"$n_s$", r"$\Omega_b$", r"$H_0$", r"$M_c^0$", r"$\nu$"], name=r'$\{c_1, \ldots, c_4\}$ '+'40 filters no standardization plus BatchNorm', shade=True, shade_alpha=0.7, color='green', linestyle='-', linewidth=2.0, kde=False)
c.add_chain(samples, parameters=["$\Omega_m$", r"$\sigma_8$", r"$w_0$", r"$n_s$", r"$\Omega_b$", r"$H_0$", r"$M_c^0$", r"$\nu$"], name=r'$\{c_1, c_2\}$ '+'CNN MSE compression, NLE on both grids', shade=True, shade_alpha=0.1, color='orange', linestyle='-', linewidth=2.0, kde=False)


fig = c.plotter.plot(figsize=(12, 12), truth=[0.26, 0.84, -1, 0.9649, 0.0493, 67.3, 13.82, 0.0], legend=True)

plt.show()

In [ ]:
samples.std(axis=0)

In [ ]:
c.analysis.get_summary()